# Phase 4: 모델 학습 — XGBoost & LSTM

## 이 노트북에서 하는 것
머신러닝 + 딥러닝 모델로 베이스라인을 뛰어넘는 성능을 달성합니다.

---
### XGBoost란?
- eXtreme Gradient Boosting — 여러 약한 결정트리를 순차적으로 쌓아 강력한 모델 생성
- 테이블형 데이터(기술지표 등)에서 **현재 최강의 ML 알고리즘 중 하나**
- LSTM보다 빠르고 해석 쉬움 (특성 중요도 시각화 가능)
- 고가/저가를 **직접** 예측 가능 (멀티-타겟)

### LSTM이란?
- Long Short-Term Memory — 순환신경망(RNN)의 진화형
- **시간 순서가 있는 데이터** (주가 시계열)에 특화된 딥러닝 모델
- 과거 N일의 패턴을 기억하면서 내일을 예측
- 단기 비선형 패턴 포착에 강함

```
[오늘-59일] → [오늘-58일] → ... → [어제] → LSTM → [내일 예측]
```

In [ ]:
!pip install "pandas==2.2.2" xgboost scikit-learn tensorflow -q
!apt-get install -y fonts-nanum -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
import os
import joblib

import xgboost as xgb
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')

# ── 한글 폰트 설정 ──────────────────────────────────────────
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
# ────────────────────────────────────────────────────────────

plt.rcParams['figure.figsize'] = (14, 5)
tf.random.set_seed(42)
np.random.seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'GPU 사용 가능: {len(gpus) > 0} — {gpus}')

train = pd.read_csv('/content/data/processed/train.csv', index_col=0, parse_dates=True)
val   = pd.read_csv('/content/data/processed/val.csv',   index_col=0, parse_dates=True)
test  = pd.read_csv('/content/data/processed/test.csv',  index_col=0, parse_dates=True)
print(f'학습: {len(train)} / 검증: {len(val)} / 테스트: {len(test)}')
print('한글 폰트: NanumGothic')

In [ ]:
# 공통 함수
def evaluate(y_true, y_pred, model_name, target_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2   = r2_score(y_true, y_pred)
    print(f'[{model_name}] {target_name:6s} | RMSE: {rmse:.3f} | MAPE: {mape:.2f}% | R2: {r2:.4f}')
    return {'model': model_name, 'target': target_name,
            'RMSE': round(rmse,3), 'MAPE(%)': round(mape,2), 'R2': round(r2,4)}

all_results = []

# 특성 선택 (타겟 컬럼 제외)
TARGET_COLS  = ['target_high', 'target_low', 'target_close', 'target_return']
FEATURE_COLS = [c for c in train.columns if c not in TARGET_COLS]
print(f'사용 특성 수: {len(FEATURE_COLS)}')

In [ ]:
# ===========================================
# XGBoost — 고가/저가 직접 예측
# ===========================================

# 스케일링 불필요 (트리 기반 모델은 스케일 무관)
X_train = train[FEATURE_COLS].fillna(0)
X_val   = val[FEATURE_COLS].fillna(0)
X_test  = test[FEATURE_COLS].fillna(0)

# 고가/저가 동시 예측 (MultiOutput)
y_train_hl = train[['target_high', 'target_low']]
y_val_hl   = val[['target_high', 'target_low']]
y_test_hl  = test[['target_high', 'target_low']]

xgb_params = {
    'n_estimators': 500,
    'max_depth': 6,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,   # L1 정규화
    'reg_lambda': 1.0,  # L2 정규화
    'random_state': 42,
    'n_jobs': -1
}

# 고가 모델
xgb_high = xgb.XGBRegressor(**xgb_params)
xgb_high.fit(X_train, y_train_hl['target_high'],
             eval_set=[(X_val, y_val_hl['target_high'])],
             verbose=100)

# 저가 모델
xgb_low = xgb.XGBRegressor(**xgb_params)
xgb_low.fit(X_train, y_train_hl['target_low'],
            eval_set=[(X_val, y_val_hl['target_low'])],
            verbose=100)

xgb_high_pred = xgb_high.predict(X_test)
xgb_low_pred  = xgb_low.predict(X_test)

print('\n--- XGBoost 성능 ---')
all_results.append(evaluate(test['target_high'].values, xgb_high_pred, 'XGBoost', 'High'))
all_results.append(evaluate(test['target_low'].values,  xgb_low_pred,  'XGBoost', 'Low'))

In [ ]:
# XGBoost 특성 중요도 시각화 (발표용 핵심 차트)
# "어떤 지표가 예측에 가장 큰 영향을 미쳤나?" 를 보여줍니다

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
for ax, model, title in [(axes[0], xgb_high, '고가 예측'), (axes[1], xgb_low, '저가 예측')]:
    importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
    top20 = importances.nlargest(20)
    top20.sort_values().plot.barh(ax=ax, color='#26a69a')
    ax.set_title(f'XGBoost 특성 중요도 — {title} (Top 20)')
    ax.set_xlabel('중요도')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/content/xgboost_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# 예측 vs 실제 산점도
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, pred, actual, title in [
    (axes[0], xgb_high_pred, test['target_high'].values, '고가'),
    (axes[1], xgb_low_pred,  test['target_low'].values,  '저가')
]:
    ax.scatter(actual, pred, alpha=0.4, s=15, color='dodgerblue')
    lims = [min(actual.min(), pred.min()), max(actual.max(), pred.max())]
    ax.plot(lims, lims, 'r--', lw=1, label='완벽한 예측선')
    ax.set_xlabel(f'실제 {title}')
    ax.set_ylabel(f'예측 {title}')
    ax.set_title(f'XGBoost {title} — 예측 vs 실제')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/xgboost_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===========================================
# LSTM — 시계열 딥러닝
#
# 데이터 준비: LOOKBACK일의 과거를 보고 내일 예측
# 예: LOOKBACK=60 → 60일치 데이터를 입력, 1일치 예측
# ===========================================

LOOKBACK = 60  # 과거 60거래일(약 3개월)을 입력

# RobustScaler: 이상치에 강한 정규화 (중앙값 기반)
# 주가는 이상치가 많아서 MinMaxScaler보다 RobustScaler가 더 적합
scaler_X = RobustScaler()
scaler_y = RobustScaler()

train_val_combined = pd.concat([train, val])
X_all = train_val_combined[FEATURE_COLS].fillna(0).values
y_all = train_val_combined[['target_high', 'target_low']].values

X_all_scaled = scaler_X.fit_transform(X_all)
y_all_scaled = scaler_y.fit_transform(y_all)
X_test_scaled = scaler_X.transform(X_test.fillna(0).values)
y_test_scaled = scaler_y.transform(test[['target_high', 'target_low']].values)

def create_sequences(X, y, lookback):
    """(samples, lookback, features) 형태의 3D 배열 생성"""
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

# 학습/검증 분할 (train 70%, val 30%)
split = int(len(X_all_scaled) * 0.85)
X_seq_all, y_seq_all = create_sequences(X_all_scaled, y_all_scaled, LOOKBACK)
X_tr_seq = X_seq_all[:split - LOOKBACK]
y_tr_seq = y_seq_all[:split - LOOKBACK]
X_vl_seq = X_seq_all[split - LOOKBACK:]
y_vl_seq = y_seq_all[split - LOOKBACK:]

# 테스트용 시퀀스 (train+val 마지막 LOOKBACK개 + test)
full_X = np.vstack([X_all_scaled, X_test_scaled])
full_y = np.vstack([y_all_scaled, y_test_scaled])
_, y_test_seq_true = create_sequences(full_y, full_y, LOOKBACK)
X_test_seq, _ = create_sequences(full_X, full_y, LOOKBACK)
X_test_seq_only = X_test_seq[-len(test):]

print(f'학습 시퀀스: {X_tr_seq.shape}')
print(f'검증 시퀀스: {X_vl_seq.shape}')
print(f'테스트 시퀀스: {X_test_seq_only.shape}')

In [ ]:
# LSTM 모델 구조 정의
# Stacked LSTM: 2층으로 쌓아서 더 복잡한 패턴 학습
# Dropout: 과적합(overfitting) 방지
# BatchNormalization: 학습 안정화

n_features = X_tr_seq.shape[2]

model_lstm = Sequential([
    Input(shape=(LOOKBACK, n_features)),
    LSTM(128, return_sequences=True),
    Dropout(0.2),
    BatchNormalization(),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dense(2)  # 출력: [예측 고가, 예측 저가]
])

model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='huber'  # Huber loss: MSE보다 이상치에 강건함
)
model_lstm.summary()

In [ ]:
# LSTM 학습
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-5)
]

history_lstm = model_lstm.fit(
    X_tr_seq, y_tr_seq,
    validation_data=(X_vl_seq, y_vl_seq),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

# 학습 곡선
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_lstm.history['loss'], label='학습 Loss')
ax.plot(history_lstm.history['val_loss'], label='검증 Loss')
ax.set_title('LSTM 학습 곡선')
ax.set_xlabel('Epoch')
ax.set_ylabel('Huber Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/lstm_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# LSTM 예측 및 역변환 (스케일 복원)
lstm_pred_scaled = model_lstm.predict(X_test_seq_only)
lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled)

lstm_high_pred = lstm_pred[:, 0]
lstm_low_pred  = lstm_pred[:, 1]

# 테스트 인덱스 정렬
test_subset = test.iloc[-len(lstm_high_pred):]

print('\n--- LSTM 성능 ---')
all_results.append(evaluate(test_subset['target_high'].values, lstm_high_pred, 'LSTM', 'High'))
all_results.append(evaluate(test_subset['target_low'].values,  lstm_low_pred,  'LSTM', 'Low'))

# 예측 vs 실제 시각화
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_subset.index, test_subset['High'], label='실제 고가', color='#26a69a', lw=1.5)
ax.plot(test_subset.index, test_subset['Low'],  label='실제 저가', color='#ef5350', lw=1.5)
ax.plot(test_subset.index, lstm_high_pred, label='LSTM 예측 고가', color='#26a69a', lw=1.5, ls='--', alpha=0.7)
ax.plot(test_subset.index, lstm_low_pred,  label='LSTM 예측 저가', color='#ef5350', lw=1.5, ls='--', alpha=0.7)
ax.fill_between(test_subset.index, lstm_low_pred, lstm_high_pred, alpha=0.1, color='#ffd700', label='예측 범위')
ax.set_title('LSTM 예측 결과 — 내일 고가/저가 (테스트셋)')
ax.set_ylabel('가격 (USD)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/lstm_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 결과 저장
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

pd.DataFrame(all_results).to_csv('/content/results/ml_results.csv', index=False)

# XGBoost 모델 저장
joblib.dump(xgb_high, '/content/models/xgb_high.pkl')
joblib.dump(xgb_low,  '/content/models/xgb_low.pkl')
joblib.dump(scaler_X, '/content/models/scaler_X.pkl')
joblib.dump(scaler_y, '/content/models/scaler_y.pkl')

# LSTM 모델 저장
model_lstm.save('/content/models/lstm_model.h5')

# 예측값 저장 (백테스트용)
for name, high, low, idx in [
    ('xgboost', xgb_high_pred, xgb_low_pred, test.index),
    ('lstm', lstm_high_pred, lstm_low_pred, test_subset.index)
]:
    actual = test if name == 'xgboost' else test_subset
    pd.DataFrame({
        'date': idx, 'pred_high': high, 'pred_low': low,
        'actual_high': actual['target_high'].values,
        'actual_low': actual['target_low'].values,
        'actual_close': actual['target_close'].values,
        'actual_open': actual['Open'].values
    }).to_csv(f'/content/results/{name}_predictions.csv', index=False)

print('모델 및 결과 저장 완료!')
print('\n다음: 05_model_tft.ipynb (Temporal Fusion Transformer)')